# Statistical Arbitrage Example

This notebook demonstrates how to use statistical arbitrage strategies to implement pairs trading, mean reversion, and cointegration analysis.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import asyncio
from statsmodels.tsa.stattools import coint, adfuller
from sklearn.linear_model import LinearRegression

from src.strategies.statistical_arbitrage import (
    StatisticalArbitrage,
    StatArbType,
    PairAnalysis,
    TradingSignal
)
from src.data.binance_fetcher import BinanceFetcher

## Configuration

Load and configure statistical arbitrage settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize trader
trader = StatisticalArbitrage(
    config=config,
    exchanges=['binance', 'kraken'],
    lookback_window=100,
    zscore_threshold=2.0,
    min_half_life=1.0,
    max_half_life=100.0,
    min_correlation=0.5,
    max_position=1.0,
    risk_limit=0.02
)

print("Trader initialized with:")
print(f"Lookback window: {trader.lookback_window}")
print(f"Z-score threshold: {trader.zscore_threshold}")
print(f"Min half-life: {trader.min_half_life}")
print(f"Max half-life: {trader.max_half_life}")
print(f"Min correlation: {trader.min_correlation}")

## Data Analysis

Analyze historical price data and identify trading opportunities.

In [ ]:
async def analyze_pairs():
    """Analyze trading pairs."""
    # Define symbols
    symbols = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'ADA/USDT', 'DOT/USDT']
    
    # Fetch price history
    prices = await trader.fetch_price_history(symbols)
    
    # Calculate correlations
    correlations = prices.corr()
    
    # Plot correlation matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlations, annot=True, cmap='RdYlBu')
    plt.title('Asset Correlations')
    plt.show()
    
    # Analyze pairs
    pairs = []
    for i in range(len(symbols)):
        for j in range(i+1, len(symbols)):
            pair = trader.analyze_pair(
                prices[symbols[i]],
                prices[symbols[j]]
            )
            if pair:
                pairs.append(pair)
    
    print(f"\nFound {len(pairs)} tradeable pairs:")
    for pair in pairs:
        print(f"\n{pair.asset_a} - {pair.asset_b}:")
        print(f"Correlation: {pair.correlation:.4f}")
        print(f"Hedge ratio: {pair.hedge_ratio:.4f}")
        print(f"Half-life: {pair.half_life:.2f}")
        print(f"Z-score: {pair.zscore:.2f}")
    
    return prices, pairs

# Analyze pairs
prices, pairs = await analyze_pairs()

## Pair Analysis

Analyze individual trading pairs in detail.

In [ ]:
def analyze_pair_details(pair: PairAnalysis, prices: pd.DataFrame):
    """Analyze pair details."""
    # Plot price series
    plt.figure(figsize=(15, 10))
    
    # Normalized prices
    plt.subplot(2, 2, 1)
    norm_a = prices[pair.asset_a] / prices[pair.asset_a].iloc[0]
    norm_b = prices[pair.asset_b] / prices[pair.asset_b].iloc[0]
    plt.plot(norm_a, label=pair.asset_a)
    plt.plot(norm_b, label=pair.asset_b)
    plt.title('Normalized Prices')
    plt.legend()
    
    # Spread
    plt.subplot(2, 2, 2)
    plt.plot(pair.spread)
    plt.axhline(y=pair.spread.mean(), color='r', linestyle='--')
    plt.axhline(y=pair.spread.mean() + 2*pair.spread.std(),
                color='g', linestyle='--')
    plt.axhline(y=pair.spread.mean() - 2*pair.spread.std(),
                color='g', linestyle='--')
    plt.title('Price Spread')
    
    # Residuals
    plt.subplot(2, 2, 3)
    plt.plot(pair.residuals)
    plt.title('Regression Residuals')
    
    # Z-score history
    plt.subplot(2, 2, 4)
    zscore = (pair.spread - pair.spread.mean()) / pair.spread.std()
    plt.plot(zscore)
    plt.axhline(y=2, color='r', linestyle='--')
    plt.axhline(y=-2, color='r', linestyle='--')
    plt.title('Z-score')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Pair Statistics:")
    print(f"Correlation: {pair.correlation:.4f}")
    print(f"Cointegration p-value: {pair.cointegration_pvalue:.4f}")
    print(f"Half-life: {pair.half_life:.2f}")
    print(f"Current z-score: {pair.zscore:.2f}")
    print(f"Hurst exponent: {pair.metrics['hurst_exponent']:.4f}")
    print(f"ADF p-value: {pair.metrics['adf_pvalue']:.4f}")

# Analyze best pair
if pairs:
    best_pair = min(pairs, key=lambda p: p.cointegration_pvalue)
    analyze_pair_details(best_pair, prices)

## Mean Reversion Analysis

Analyze mean reversion opportunities.

In [ ]:
def analyze_mean_reversion(prices: pd.DataFrame):
    """Analyze mean reversion opportunities."""
    results = []
    
    for symbol in prices.columns:
        # Generate signal
        signal = trader.generate_mean_reversion_signal(
            prices[symbol]
        )
        
        if signal:
            results.append({
                'symbol': symbol,
                'direction': signal.direction[0],
                'strength': signal.confidence,
                'zscore': signal.entry_zscore
            })
    
    if results:
        print("Mean Reversion Opportunities:")
        for result in results:
            print(f"\n{result['symbol']}:")
            print(f"Direction: {'Long' if result['direction'] > 0 else 'Short'}")
            print(f"Signal Strength: {result['strength']:.2f}")
            print(f"Z-score: {result['zscore']:.2f}")
    
        # Plot opportunities
        plt.figure(figsize=(15, 5))
        
        for result in results:
            symbol = result['symbol']
            prices_norm = prices[symbol] / prices[symbol].iloc[0]
            plt.plot(prices_norm, label=symbol)
        
        plt.title('Mean Reversion Opportunities')
        plt.legend()
        plt.show()

# Analyze mean reversion
analyze_mean_reversion(prices)

## Trading Simulation

Simulate statistical arbitrage trading.

In [ ]:
async def simulate_trading(duration: int = 30):
    """Simulate trading strategy."""
    print(f"Running strategy for {duration} seconds...")
    
    symbols = list(prices.columns)
    
    # Run strategy
    try:
        async def stop_after_delay():
            await asyncio.sleep(duration)
            raise KeyboardInterrupt
        
        await asyncio.gather(
            trader.run(symbols, interval=1.0),
            stop_after_delay()
        )
    except KeyboardInterrupt:
        pass
    
    # Analyze results
    metrics = trader.get_performance_metrics()
    
    print("\nPerformance Metrics:")
    print(f"Total P&L: ${metrics['total_pnl']:,.2f}")
    print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"Win Rate: {metrics['win_rate']*100:.1f}%")
    print(f"Number of Trades: {metrics['num_trades']}")
    
    # Plot results
    plt.figure(figsize=(15, 10))
    
    # P&L
    plt.subplot(2, 2, 1)
    pnl = pd.Series([t['pnl'] for t in trader.trades])
    plt.plot(pnl.cumsum())
    plt.title('Cumulative P&L')
    
    # Positions
    plt.subplot(2, 2, 2)
    positions = pd.DataFrame(trader.positions.items(),
                           columns=['symbol', 'size'])
    positions.plot(kind='bar', x='symbol', y='size')
    plt.title('Current Positions')
    plt.xticks(rotation=45)
    
    # Signal distribution
    plt.subplot(2, 2, 3)
    signal_types = [s.type.value for s in trader.signals]
    pd.Series(signal_types).value_counts().plot(kind='bar')
    plt.title('Signal Distribution')
    plt.xticks(rotation=45)
    
    # Return distribution
    plt.subplot(2, 2, 4)
    sns.histplot(pnl)
    plt.title('Return Distribution')
    
    plt.tight_layout()
    plt.show()

# Run simulation
await simulate_trading()